# Getting Started with Grilly

**A PyTorch-like neural network framework powered by Vulkan compute shaders.**

This notebook walks you through:
1. Installing grilly
2. Checking your GPU/Vulkan backend status
3. Creating a simple 2-layer MLP
4. Running a forward pass
5. Running a backward pass with an optimizer step

Grilly runs on both GPU (Vulkan) and CPU (numpy fallback), so this notebook
will work on any machine.

## 1. Installation

Install grilly from PyPI. If you already have it installed locally via
`pip install -e .`, you can skip this cell.

In [ ]:
# Uncomment the line below to install grilly from PyPI
# !pip install grilly

## 2. Version and Backend Check

Let's verify that grilly is installed correctly and check whether the Vulkan
GPU backend is available. If Vulkan is not found, grilly falls back to numpy
automatically -- all code in this notebook runs either way.

In [ ]:
import grilly
from grilly import nn
from grilly.nn import Variable, tensor
import numpy as np

print(f"grilly version : {grilly.__version__}")

# Check Vulkan backend availability
try:
    from grilly._bridge import is_vulkan_available
    vulkan_ok = is_vulkan_available()
except (ImportError, AttributeError):
    vulkan_ok = False

if vulkan_ok:
    print("Backend        : Vulkan GPU (accelerated)")
else:
    print("Backend        : numpy CPU (fallback)")
    print("Tip: Install Vulkan drivers for GPU acceleration.")

print(f"numpy version  : {np.__version__}")

## 3. Build a Simple 2-Layer MLP

We will build a small multi-layer perceptron (MLP) suitable for MNIST-style
classification:

```
Input (784) -> Linear(784, 256) -> ReLU -> Linear(256, 10) -> Output (10)
```

Grilly's `nn.Sequential` works just like PyTorch -- you stack layers in order
and the framework handles the forward pass.

In [ ]:
# Define a simple 2-layer MLP
model = nn.Sequential(
    nn.Linear(784, 256),   # Hidden layer: 784 input features -> 256 hidden units
    nn.ReLU(),              # Activation function
    nn.Linear(256, 10),    # Output layer: 256 hidden units -> 10 classes
)

print("Model architecture:")
print(model)
print()

# Count parameters
total_params = sum(p.data.size for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"  Linear(784, 256): 784*256 + 256 = {784*256 + 256:,} params")
print(f"  Linear(256, 10) : 256*10  + 10  = {256*10 + 10:,} params")

## 4. Forward Pass on Random Data

Let's create a batch of random "images" (flattened 28x28 = 784 pixels) and
push them through the model. The output will be 10 raw logits per sample --
one per class.

In [ ]:
# Create a batch of 8 random inputs (simulating flattened 28x28 images)
batch_size = 8
x_np = np.random.randn(batch_size, 784).astype(np.float32)
x = Variable(x_np)

print(f"Input shape : {x.data.shape}")
print(f"Input dtype : {x.data.dtype}")
print()

# Forward pass
logits = model(x)

print(f"Output shape: {logits.data.shape}  (batch_size={batch_size}, num_classes=10)")
print(f"Output dtype: {logits.data.dtype}")
print()
print("First sample logits (raw, unnormalized):")
print(np.round(logits.data[0], 4))

## 5. Backward Pass and Optimizer Step

Now let's do a complete training step:
1. Compute a simple MSE loss against random targets
2. Run backpropagation (`.backward()`)
3. Update weights with AdamW

This demonstrates grilly's autograd engine in action.

In [ ]:
from grilly.optim import AdamW
import grilly.functional as F

# Create optimizer
optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

# Generate random targets (one-hot encoded for 10 classes)
target_classes = np.random.randint(0, 10, size=(batch_size,))
targets_np = np.zeros((batch_size, 10), dtype=np.float32)
targets_np[np.arange(batch_size), target_classes] = 1.0
targets = Variable(targets_np)

# Forward pass
logits = model(x)

# Compute MSE loss: mean squared error between logits and one-hot targets
diff = logits - targets
loss = (diff * diff).sum() / batch_size

print(f"Loss before step: {loss.data:.4f}")

# Backward pass -- computes gradients for all parameters
loss.backward()

# Check that gradients were computed
first_layer = model.layers[0]  # Linear(784, 256)
print(f"Weight grad shape: {first_layer.weight.grad.shape}")
print(f"Weight grad mean : {first_layer.weight.grad.mean():.6f}")
print(f"Weight grad std  : {first_layer.weight.grad.std():.6f}")

# Optimizer step -- updates all parameters using AdamW
optimizer.step()
optimizer.zero_grad()

# Verify the loss decreased after the update
logits_after = model(x)
diff_after = logits_after - targets
loss_after = (diff_after * diff_after).sum() / batch_size

print(f"\nLoss after step : {loss_after.data:.4f}")
print(f"Loss decreased  : {loss_after.data < loss.data}")

## 6. Summary

In this notebook you learned:

- **Installation**: `pip install grilly` gives you the framework
- **Backend detection**: Grilly automatically uses Vulkan GPU if available,
  otherwise falls back to numpy on CPU
- **Model building**: `nn.Sequential`, `nn.Linear`, and `nn.ReLU` work like
  their PyTorch counterparts
- **Forward pass**: Wrap numpy arrays in `Variable` and call the model
- **Backward pass**: Call `.backward()` on the loss to compute gradients,
  then `optimizer.step()` to update weights

### Next Steps

- **Notebook 02**: Complete training loop with loss curves and decision boundaries
- **Notebook 03**: Spiking neural networks with LIF neurons
- **Notebook 04**: Vector symbolic architectures (bind, bundle, similarity)
- **Notebook 05**: Attention mechanisms and transformer blocks